In [2]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import EfficientNet_V2_M_Weights
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [3]:
DATA_DIR       = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split"
OUTPUT_DIR     = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m"
IMG_SIZE       = 224        # RTX 3050 4GB için 224 — 480 OOM yaratır
BATCH_SIZE     = 8          # Fine-tune sırasında tüm model açık olduğu için 8 güvenli
ACCUM_STEPS    = 4          # Gradient accumulation: efektif batch = 8x4 = 32
NUM_WORKERS    = 0          # Windows CMD için 0 olmalı
PIN_MEMORY     = True
USE_AMP        = True       # Mixed precision (FP16) — VRAM'ı ~yarıya düşürür
 
# Epoch sayıları
FREEZE_EPOCHS  = 5          # Aşama 1: Backbone dondurulmuş
FINETUNE_EPOCHS = 5         # Aşama 2: Fine-tune
 
# Learning rate
FREEZE_LR      = 1e-3       # Aşama 1 LR
FINETUNE_LR    = 1e-5       # Aşama 2 LR (çok düşük!)
 
# Augmentation modu: "both" | "mixup" | "cutmix" | "none"
AUGMENTATION_MODE = "both"
 
# MixUp / CutMix parametreleri
MIXUP_ALPHA    = 0.4
CUTMIX_ALPHA   = 1.0
MIXUP_PROB     = 0.5        # "both" modunda MixUp seçilme olasılığı
 
SEED           = 42
SUBSET_RATIO   = 0.4        # Her epoch train setinin %40'ını kullan (hız için)

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
set_seed(SEED)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Kullanılan cihaz: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

✅ Kullanılan cihaz: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   VRAM: 4.3 GB


In [6]:
# ─────────────────────────────────────────────
# TRANSFORM
# EfficientNetV2-M için resmi öneri: 480x480, ImageNet normalizasyonu
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
 
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [7]:
# ─────────────────────────────────────────────
# SAFE DATASET - Hatalı dosyaları atlar
# ─────────────────────────────────────────────
from torchvision.datasets import ImageFolder
from torchvision.datasets.folder import default_loader
 
class SafeImageFolder(ImageFolder):
    """Açılamayan dosyaları sessizce atlayan ImageFolder"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print(f"   🔍 Dosyalar doğrulanıyor: {self.root}")
        valid_samples = []
        skipped = 0
        for path, label in tqdm(self.samples, desc="   Taranıyor", leave=False):
            try:
                with open(path, 'rb') as f:
                    f.read(10)
                valid_samples.append((path, label))
            except (FileNotFoundError, OSError):
                skipped += 1
        if skipped > 0:
            print(f"   ⚠️  {skipped} dosya atlandı (bulunamadı/açılamadı)")
        self.samples = valid_samples
        self.targets = [s[1] for s in valid_samples]
        print(f"   ✅ Geçerli dosya sayısı: {len(self.samples)}")

In [8]:
# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────
print("\n📂 Veri seti yükleniyor...")
train_dataset = SafeImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = SafeImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_test_transform)
test_dataset  = SafeImageFolder(os.path.join(DATA_DIR, "test"),  transform=val_test_transform)
 
NUM_CLASSES = len(train_dataset.classes)
print(f"   Sınıf sayısı : {NUM_CLASSES}")
print(f"   Train        : {len(train_dataset)} görsel")
print(f"   Validation   : {len(val_dataset)} görsel")
print(f"   Test         : {len(test_dataset)} görsel")


📂 Veri seti yükleniyor...
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\train


   ⚠️  1 dosya atlandı (bulunamadı/açılamadı)
   ✅ Geçerli dosya sayısı: 34019
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\val


   ✅ Geçerli dosya sayısı: 2043
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979
   Sınıf sayısı : 47
   Train        : 34019 görsel
   Validation   : 2043 görsel
   Test         : 1979 görsel


In [9]:
# ─────────────────────────────────────────────
# WEIGHTED RANDOM SAMPLER
# ─────────────────────────────────────────────
def get_weighted_sampler(dataset):
    targets = [s[1] for s in dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )
    return sampler
 
sampler = get_weighted_sampler(train_dataset)
 
subset_size = int(len(train_dataset) * SUBSET_RATIO)
subset_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor([sampler.weights[i] for i in range(len(train_dataset))]),
    num_samples=subset_size,
    replacement=True
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=subset_sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f"   Her epoch: {subset_size} görsel / {subset_size//BATCH_SIZE} batch (toplam {len(train_dataset)} görselin %{int(SUBSET_RATIO*100)}'i)")
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

   Her epoch: 13607 görsel / 1700 batch (toplam 34019 görselin %40'i)


In [10]:
# ─────────────────────────────────────────────
# MixUp / CutMix Fonksiyonları
# ─────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam
 
def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [11]:
# ─────────────────────────────────────────────
# MODEL - EfficientNetV2-M (torchvision)
# https://docs.pytorch.org/vision/main/models/generated/torchvision.models.efficientnet_v2_m.html
# ─────────────────────────────────────────────
print("\n🔧 EfficientNetV2-M modeli yükleniyor (torchvision)...")
model = models.efficientnet_v2_m(weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1)
 
# EfficientNetV2-M classifier katmanını değiştir
# Orijinal: model.classifier = Sequential(Dropout, Linear(1280, 1000))
in_features = model.classifier[1].in_features  # 1280
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)
 
model = model.to(device)
print(f"   Model yüklendi. Parametre sayısı: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Classifier in_features: {in_features} → out: {NUM_CLASSES}")


🔧 EfficientNetV2-M modeli yükleniyor (torchvision)...
   Model yüklendi. Parametre sayısı: 52,918,563
   Classifier in_features: 1280 → out: 47


In [12]:
# ─────────────────────────────────────────────
# FREEZE / UNFREEZE Yardımcıları
# EfficientNetV2-M mimarisi: model.features (backbone) + model.classifier (head)
# ─────────────────────────────────────────────
def freeze_backbone(model):
    """Sadece classifier katmanını eğitilebilir bırak, features'ı dondur"""
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"   ❄️  Backbone (features) donduruldu.")
    print(f"       Eğitilebilir: {trainable:,} / Toplam: {total:,} parametre")
 
def unfreeze_all(model):
    """Tüm parametreleri eğitilebilir yap"""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔥 Tüm model açıldı. Eğitilebilir parametre: {trainable:,}")

In [13]:
# ─────────────────────────────────────────────
# METRİK HESAPLAMA
# ─────────────────────────────────────────────
def evaluate(model, loader, criterion, phase="val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
 
    # Eval öncesi CUDA cache temizle
    torch.cuda.empty_cache()
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            # Her batch sonrası tensörleri serbest bırak
            del images, labels, outputs, loss
 
    torch.cuda.empty_cache()
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
 
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels

In [14]:
# ─────────────────────────────────────────────
# CONFUSION MATRIX KAYDET
# ─────────────────────────────────────────────
def save_confusion_matrix(labels, preds, class_names, save_path, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 6})
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Tahmin", fontsize=10)
    ax.set_ylabel("Gerçek", fontsize=10)
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Confusion matrix kaydedildi: {save_path}")
 

In [15]:
# ─────────────────────────────────────────────
# METRİK GRAFİĞİ KAYDET
# ─────────────────────────────────────────────
def save_metrics_plot(history, save_path, title="Eğitim Grafiği"):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14)
 
    metrics = [
        ('train_loss', 'val_loss', 'Loss'),
        ('train_acc',  'val_acc',  'Accuracy'),
        ('train_f1',   'val_f1',   'F1 Score'),
        ('train_prec', 'val_prec', 'Precision'),
        ('train_rec',  'val_rec',  'Recall'),
    ]
    for i, (tr_key, val_key, label) in enumerate(metrics):
        ax = axes[i // 3][i % 3]
        ax.plot(epochs, history[tr_key], 'b-o', label='Train')
        ax.plot(epochs, history[val_key], 'r-o', label='Val')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True)
    axes[1][2].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"   💾 Grafik kaydedildi: {save_path}")

In [16]:
# ─────────────────────────────────────────────
# EĞİTİM FONKSİYONU
# Her epoch sonunda train ve val confusion matrix de kaydedilir
# ─────────────────────────────────────────────
def train_phase(model, train_loader, val_loader, optimizer, scheduler,
                criterion, num_epochs, phase_name, aug_mode, class_names):
 
    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc',
                                 'train_f1','val_f1','train_prec','val_prec',
                                 'train_rec','val_rec']}
    best_val_f1 = 0.0
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{phase_name}.pth")
 
    cm_dir = os.path.join(OUTPUT_DIR, f"confusion_matrices_{phase_name}")
    os.makedirs(cm_dir, exist_ok=True)
 
    # AMP GradScaler — FP16 eğitim için
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
 
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []
        start = time.time()
        torch.cuda.empty_cache()
 
        skipped_batches = 0
        optimizer.zero_grad()  # Accumulation için epoch başında sıfırla
        pbar = tqdm(train_loader, desc=f"[{phase_name}] Epoch {epoch}/{num_epochs}")
 
        for batch_idx, batch_data in enumerate(pbar):
            try:
                images, labels = batch_data
                images, labels = images.to(device), labels.to(device)
 
                use_aug = aug_mode != "none"
                if use_aug:
                    if aug_mode == "both":
                        use_mixup = random.random() < MIXUP_PROB
                    elif aug_mode == "mixup":
                        use_mixup = True
                    else:
                        use_mixup = False
 
                    if use_mixup:
                        images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                    else:
                        images, y_a, y_b, lam = cutmix_data(images, labels, CUTMIX_ALPHA)
 
                # AMP ile forward pass
                with torch.autocast(device_type="cuda", enabled=USE_AMP):
                    outputs = model(images)
                    if use_aug and aug_mode != "none":
                        loss = mixed_criterion(criterion, outputs, y_a, y_b, lam)
                    else:
                        loss = criterion(outputs, labels)
                    # Accumulation için loss'u böl
                    loss = loss / ACCUM_STEPS
 
                scaler.scale(loss).backward()
 
                # Gradient accumulation: ACCUM_STEPS dolunca güncelle
                if (batch_idx + 1) % ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
 
                total_loss += loss.item() * ACCUM_STEPS  # Gerçek loss değeri
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'loss': f'{loss.item() * ACCUM_STEPS:.4f}'})
 
                # Tensörleri temizle
                del images, labels, outputs, loss
 
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"\n   ⚠️  OOM! Batch {batch_idx} atlanıyor, cache temizleniyor...")
                    torch.cuda.empty_cache()
                    optimizer.zero_grad()
                    skipped_batches += 1
                    continue
                else:
                    raise e
            except Exception:
                skipped_batches += 1
                continue
 
        if skipped_batches > 0:
            print(f"   {skipped_batches} batch atlandı")
 
        if scheduler:
            scheduler.step()
 
        torch.cuda.empty_cache()
 
        tr_loss = total_loss / len(train_loader)
        tr_acc  = accuracy_score(all_labels, all_preds)
        tr_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
 
        val_loss, val_acc, val_prec, val_rec, val_f1, val_preds, val_labels = evaluate(
            model, val_loader, criterion
        )
 
        elapsed = time.time() - start
        print(f"\n📊 [{phase_name}] Epoch {epoch}/{num_epochs} ({elapsed:.0f}s)")
        print(f"   Train → Loss:{tr_loss:.4f}  Acc:{tr_acc:.4f}  F1:{tr_f1:.4f}  Prec:{tr_prec:.4f}  Rec:{tr_rec:.4f}")
        print(f"   Val   → Loss:{val_loss:.4f}  Acc:{val_acc:.4f}  F1:{val_f1:.4f}  Prec:{val_prec:.4f}  Rec:{val_rec:.4f}")
 
        history['train_loss'].append(tr_loss);  history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc);    history['val_acc'].append(val_acc)
        history['train_f1'].append(tr_f1);      history['val_f1'].append(val_f1)
        history['train_prec'].append(tr_prec);  history['val_prec'].append(val_prec)
        history['train_rec'].append(tr_rec);    history['val_rec'].append(val_rec)
 
        train_cm_path = os.path.join(cm_dir, f"epoch{epoch:02d}_train_cm.png")
        save_confusion_matrix(
            all_labels, all_preds, class_names, train_cm_path,
            title=f"[{phase_name}] Train CM — Epoch {epoch}/{num_epochs} | F1:{tr_f1:.4f}"
        )
 
        val_cm_path = os.path.join(cm_dir, f"epoch{epoch:02d}_val_cm.png")
        save_confusion_matrix(
            val_labels, val_preds, class_names, val_cm_path,
            title=f"[{phase_name}] Val CM — Epoch {epoch}/{num_epochs} | F1:{val_f1:.4f}"
        )
 
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"   ✅ En iyi model kaydedildi (Val F1: {best_val_f1:.4f})")
 
    return history, best_model_path

In [17]:
# ─────────────────────────────────────────────
# TEST DEĞERLENDİRME
# ─────────────────────────────────────────────
def test_evaluation(model, test_loader, criterion, class_names, phase_label, aug_label):
    print(f"\n{'='*60}")
    print(f"🧪 TEST DEĞERLENDİRMESİ - {phase_label} | Aug: {aug_label}")
    print(f"{'='*60}")
    _, acc, prec, rec, f1, preds, labels = evaluate(model, test_loader, criterion, "test")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"   F1 Score  : {f1:.4f}")
 
    # Test Confusion matrix
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{phase_label}_{aug_label}.png")
    save_confusion_matrix(labels, preds, class_names, cm_path,
                          title=f"Confusion Matrix - {phase_label} | {aug_label}")
 
    # Classification report
    report = classification_report(labels, preds, target_names=class_names, zero_division=0)
    report_path = os.path.join(OUTPUT_DIR, f"classification_report_{phase_label}_{aug_label}.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"Phase: {phase_label} | Augmentation: {aug_label}\n\n")
        f.write(report)
    print(f"   💾 Classification report: {report_path}")
 
    return acc, prec, rec, f1

In [18]:
# ─────────────────────────────────────────────
# GRAD-CAM DEĞERLENDİRMESİ
# ─────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
 
GRADCAM_DIR = os.path.join(OUTPUT_DIR, "gradcam")
os.makedirs(GRADCAM_DIR, exist_ok=True)
 
SAMPLES_PER_CLASS = 2   # Her sınıftan kaç görsel
 
vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
 
norm_transform = transforms.Normalize(
    [0.485, 0.456, 0.406],
    [0.229, 0.224, 0.225]
)
 
def get_target_layer(model):
    """
    EfficientNetV2-M için son konvolüsyon bloğunu döndür.
    model.features[-1] → son MBConv/FusedMBConv bloğu
    """
    return [model.features[-1]]
 
def run_gradcam(model, model_name, dataset, class_names):
    """Verilen model için her sınıftan SAMPLES_PER_CLASS görsel üzerinde Grad-CAM uygula"""
    model.eval()
    target_layers = get_target_layer(model)
    cam = GradCAM(model=model, target_layers=target_layers)
 
    save_dir = os.path.join(GRADCAM_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
 
    # Her sınıftan örnek topla
    class_samples = {i: [] for i in range(len(class_names))}
    for path, label in dataset.samples:
        if len(class_samples[label]) < SAMPLES_PER_CLASS:
            class_samples[label].append(path)
        if all(len(v) >= SAMPLES_PER_CLASS for v in class_samples.values()):
            break
 
    print(f"\n🎨 [{model_name}] Grad-CAM oluşturuluyor...")
    for class_idx, paths in tqdm(class_samples.items(), desc=model_name):
        class_name = class_names[class_idx]
        fig, axes = plt.subplots(len(paths), 3, figsize=(12, 4 * len(paths)))
        if len(paths) == 1:
            axes = [axes]
        fig.suptitle(f"{model_name} | Sınıf: {class_name}", fontsize=11)
 
        for row, path in enumerate(paths):
            try:
                from PIL import Image as PILImage
                pil_img = PILImage.open(path).convert("RGB")
                pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
                rgb_img = np.array(pil_img).astype(np.float32) / 255.0
 
                input_tensor = norm_transform(vis_transform(pil_img)).unsqueeze(0).to(device)
 
                with torch.no_grad():
                    output = model(input_tensor)
                pred_idx = output.argmax(dim=1).item()
                pred_name = class_names[pred_idx]
                confidence = torch.softmax(output, dim=1)[0][pred_idx].item()
 
                targets = [ClassifierOutputTarget(class_idx)]
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0]
                cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
 
                is_correct = (pred_idx == class_idx)
                border_color = "green" if is_correct else "red"
                status = "✅ Doğru" if is_correct else f"❌ Yanlış → {pred_name}"
 
                axes[row][0].imshow(pil_img)
                axes[row][0].set_title("Orijinal", fontsize=9)
                axes[row][0].axis("off")
 
                axes[row][1].imshow(grayscale_cam, cmap="jet")
                axes[row][1].set_title("Grad-CAM Isı Haritası", fontsize=9)
                axes[row][1].axis("off")
 
                axes[row][2].imshow(cam_image)
                axes[row][2].set_title(
                    f"{status}\nGüven: {confidence:.2%}", fontsize=9,
                    color=border_color
                )
                axes[row][2].axis("off")
                for spine in axes[row][2].spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)
 
            except Exception as e:
                print(f"   ⚠️  {path} atlandı: {e}")
                continue
 
        plt.tight_layout()
        safe_name = class_name.replace("/", "_").replace("\\", "_")
        save_path = os.path.join(save_dir, f"{safe_name}.png")
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        plt.close()
 
    print(f"   💾 Kaydedildi: {save_dir}")

In [19]:
# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
class_names = train_dataset.classes
 
print(f"\n{'='*60}")
print(f"🚀 EĞİTİM BAŞLIYOR")
print(f"   Model          : EfficientNetV2-M (torchvision)")
print(f"   Görsel boyutu  : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Augmentation   : {AUGMENTATION_MODE}")
print(f"   Freeze Epochs  : {FREEZE_EPOCHS}")
print(f"   Finetune Epochs: {FINETUNE_EPOCHS}")
print(f"   Batch Size     : {BATCH_SIZE}")
print(f"{'='*60}\n")


🚀 EĞİTİM BAŞLIYOR
   Model          : EfficientNetV2-M (torchvision)
   Görsel boyutu  : 224x224
   Augmentation   : both
   Freeze Epochs  : 5
   Finetune Epochs: 5
   Batch Size     : 8



In [20]:
# ──────────────────────────────
# AŞAMA 1: BACKBONE FREEZE
# ──────────────────────────────
print("\n" + "="*60)
print("⚡ AŞAMA 1: BACKBONE FREEZE")
print("="*60)
freeze_backbone(model)
 
optimizer_freeze = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FREEZE_LR, weight_decay=1e-4
)
scheduler_freeze = optim.lr_scheduler.CosineAnnealingLR(optimizer_freeze, T_max=FREEZE_EPOCHS)
 
history_freeze, best_freeze_path = train_phase(
    model, train_loader, val_loader,
    optimizer_freeze, scheduler_freeze,
    criterion, FREEZE_EPOCHS,
    phase_name="freeze",
    aug_mode=AUGMENTATION_MODE,
    class_names=class_names
)
 
# Grafik kaydet
save_metrics_plot(history_freeze,
                  os.path.join(OUTPUT_DIR, "grafik_freeze.png"),
                  title=f"Aşama 1: Backbone Freeze | Aug: {AUGMENTATION_MODE}")
 
# Aşama 1 test değerlendirmesi
print("\n📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_freeze_path, map_location=device))
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)
 


⚡ AŞAMA 1: BACKBONE FREEZE
   ❄️  Backbone (features) donduruldu.
       Eğitilebilir: 60,207 / Toplam: 52,918,563 parametre


[freeze] Epoch 1/5: 100%|██████████| 1701/1701 [02:23<00:00, 11.84it/s, loss=3.5380]



📊 [freeze] Epoch 1/5 (158s)
   Train → Loss:3.0344  Acc:0.1954  F1:0.1831  Prec:0.1830  Rec:0.1936
   Val   → Loss:2.0787  Acc:0.4782  F1:0.3990  Prec:0.4486  Rec:0.4347
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch01_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch01_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.3990)


[freeze] Epoch 2/5: 100%|██████████| 1701/1701 [02:22<00:00, 11.98it/s, loss=1.4588]



📊 [freeze] Epoch 2/5 (158s)
   Train → Loss:2.6424  Acc:0.2567  F1:0.2472  Prec:0.2451  Rec:0.2561
   Val   → Loss:1.8360  Acc:0.5144  F1:0.4433  Prec:0.4792  Rec:0.4674
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch02_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch02_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.4433)


...


📊 [freeze] Epoch 3/5 (165s)
   Train → Loss:2.5140  Acc:0.2868  F1:0.2769  Prec:0.2751  Rec:0.2840
   Val   → Loss:1.8071  Acc:0.5179  F1:0.4483  Prec:0.4847  Rec:0.4684
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch03_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch03_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.4483)


[freeze] Epoch 4/5: 100%|██████████| 1701/1701 [02:18<00:00, 12.31it/s, loss=2.8333]



📊 [freeze] Epoch 4/5 (151s)
   Train → Loss:2.4849  Acc:0.2954  F1:0.2888  Prec:0.2864  Rec:0.2952
   Val   → Loss:1.7489  Acc:0.5360  F1:0.4770  Prec:0.5088  Rec:0.4992
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch04_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch04_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.4770)


[freeze] Epoch 5/5: 100%|██████████| 1701/1701 [02:04<00:00, 13.68it/s, loss=3.0987]



📊 [freeze] Epoch 5/5 (137s)
   Train → Loss:2.4828  Acc:0.2946  F1:0.2884  Prec:0.2868  Rec:0.2952
   Val   → Loss:1.7581  Acc:0.5326  F1:0.4621  Prec:0.4870  Rec:0.4859
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch05_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_freeze\epoch05_val_cm.png
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\grafik_freeze.png

📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: both
   Accuracy  : 0.5361
   Precision : 0.5059
   Recall    : 0.4864
   F1 Score  : 0.4673
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrix_Phase1_Freeze_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outpu

In [21]:
# ──────────────────────────────
# AŞAMA 2: FINE-TUNING
# ──────────────────────────────
print("\n" + "="*60)
print("🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)")
print("="*60)
unfreeze_all(model)
 
optimizer_finetune = optim.AdamW(
    model.parameters(),
    lr=FINETUNE_LR, weight_decay=1e-4
)
scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=FINETUNE_EPOCHS)
 
history_finetune, best_finetune_path = train_phase(
    model, train_loader, val_loader,
    optimizer_finetune, scheduler_finetune,
    criterion, FINETUNE_EPOCHS,
    phase_name="finetune",
    aug_mode=AUGMENTATION_MODE,
    class_names=class_names
)
 
# Grafik kaydet
save_metrics_plot(history_finetune,
                  os.path.join(OUTPUT_DIR, "grafik_finetune.png"),
                  title=f"Aşama 2: Fine-Tuning | Aug: {AUGMENTATION_MODE}")
 
# Aşama 2 test değerlendirmesi
print("\n📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_finetune_path, map_location=device))
acc2, prec2, rec2, f1_2 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase2_FineTune", aug_label=AUGMENTATION_MODE
)


🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)
   🔥 Tüm model açıldı. Eğitilebilir parametre: 52,918,563


[finetune] Epoch 1/5:   0%|          | 0/1701 [00:00<?, ?it/s]

[finetune] Epoch 1/5: 100%|██████████| 1701/1701 [10:27<00:00,  2.71it/s, loss=2.1479] 



📊 [finetune] Epoch 1/5 (662s)
   Train → Loss:2.2675  Acc:0.3470  F1:0.3407  Prec:0.3401  Rec:0.3469
   Val   → Loss:1.2724  Acc:0.6412  F1:0.5867  Prec:0.5988  Rec:0.6030
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch01_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch01_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.5867)


[finetune] Epoch 2/5: 100%|██████████| 1701/1701 [11:13<00:00,  2.53it/s, loss=0.8510]


...


📊 [finetune] Epoch 3/5 (709s)
   Train → Loss:1.9344  Acc:0.4368  F1:0.4317  Prec:0.4302  Rec:0.4360
   Val   → Loss:0.9612  Acc:0.7288  F1:0.6836  Prec:0.6907  Rec:0.6895
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch03_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch03_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.6836)


[finetune] Epoch 4/5: 100%|██████████| 1701/1701 [10:50<00:00,  2.62it/s, loss=2.4685]



📊 [finetune] Epoch 4/5 (683s)
   Train → Loss:1.8710  Acc:0.4433  F1:0.4389  Prec:0.4379  Rec:0.4429
   Val   → Loss:0.8991  Acc:0.7406  F1:0.6986  Prec:0.7072  Rec:0.7060
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch04_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch04_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.6986)


[finetune] Epoch 5/5: 100%|██████████| 1701/1701 [11:12<00:00,  2.53it/s, loss=2.1349]



📊 [finetune] Epoch 5/5 (706s)
   Train → Loss:1.8430  Acc:0.4426  F1:0.4372  Prec:0.4359  Rec:0.4407
   Val   → Loss:0.8799  Acc:0.7499  F1:0.7080  Prec:0.7155  Rec:0.7120
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch05_train_cm.png
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrices_finetune\epoch05_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.7080)
   💾 Grafik kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\grafik_finetune.png

📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase2_FineTune | Aug: both
   Accuracy  : 0.7322
   Precision : 0.7039
   Recall    : 0.6883
   F1 Score  : 0.6859
   💾 Confusion matrix kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\confusion_matrix_Phase2_FineTune_both.png
   💾 Classification

In [22]:
# ──────────────────────────────
# GRAD-CAM
# ──────────────────────────────
FREEZE_PATH   = os.path.join(OUTPUT_DIR, "best_freeze.pth")
FINETUNE_PATH = os.path.join(OUTPUT_DIR, "best_finetune.pth")
 
gradcam_dataset = SafeImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=val_test_transform
)
 
print("\n" + "="*60)
print("🔍 GRAD-CAM BAŞLIYOR")
print("="*60)
print(f"   Her sınıftan {SAMPLES_PER_CLASS} görsel x {len(train_dataset.classes)} sınıf")
 
print("\n⚡ Backbone Freeze modeli yükleniyor...")
model.load_state_dict(torch.load(FREEZE_PATH, map_location=device))
run_gradcam(model, "Phase1_BackboneFreeze", gradcam_dataset, train_dataset.classes)
 
print("\n🔥 Fine-Tune modeli yükleniyor...")
model.load_state_dict(torch.load(FINETUNE_PATH, map_location=device))
run_gradcam(model, "Phase2_FineTune", gradcam_dataset, train_dataset.classes)
 
print("\n" + "="*60)
print(f"✅ GRAD-CAM TAMAMLANDI! → {GRADCAM_DIR}")
print("="*60)

   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979

🔍 GRAD-CAM BAŞLIYOR
   Her sınıftan 2 görsel x 47 sınıf

⚡ Backbone Freeze modeli yükleniyor...

🎨 [Phase1_BackboneFreeze] Grad-CAM oluşturuluyor...


Phase1_BackboneFreeze: 100%|██████████| 47/47 [01:15<00:00,  1.61s/it]


   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\gradcam\Phase1_BackboneFreeze

🔥 Fine-Tune modeli yükleniyor...

🎨 [Phase2_FineTune] Grad-CAM oluşturuluyor...


Phase2_FineTune: 100%|██████████| 47/47 [01:09<00:00,  1.49s/it]

   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\gradcam\Phase2_FineTune

✅ GRAD-CAM TAMAMLANDI! → C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\gradcam


In [23]:
# ──────────────────────────────
# KARŞILAŞTIRMA TABLOSU
# ──────────────────────────────
print("\n" + "="*60)
print("📈 SONUÇ KARŞILAŞTIRMASI")
print("="*60)
print(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-"*60)
print(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}")
print(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}")
print("="*60)
 
# Özet dosyası kaydet
summary_path = os.path.join(OUTPUT_DIR, f"summary_{AUGMENTATION_MODE}.txt")
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"EfficientNetV2-M Eğitim Özeti\n")
    f.write(f"Augmentation: {AUGMENTATION_MODE}\n")
    f.write(f"Görsel boyutu: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"{'='*60}\n")
    f.write(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}\n")
    f.write(f"{'-'*60}\n")
    f.write(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}\n")
    f.write(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}\n")
 
print(f"\n✅ Tüm sonuçlar kaydedildi: {OUTPUT_DIR}")
print(f"   Özet dosyası: {summary_path}")
print("\n🎉 EĞİTİM TAMAMLANDI!")


📈 SONUÇ KARŞILAŞTIRMASI
Aşama                               Acc     Prec      Rec       F1
------------------------------------------------------------
Phase 1 (Freeze)                 0.5361   0.5059   0.4864   0.4673
Phase 2 (Fine-Tune)              0.7322   0.7039   0.6883   0.6859

✅ Tüm sonuçlar kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m
   Özet dosyası: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_efficientnetv2m\summary_both.txt

🎉 EĞİTİM TAMAMLANDI!
